# Deep Learning Project ( Latent Diffusion Models )

## Base Setup

In [40]:
import torch
import sys

print(f"Python Version: {sys.version}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("WARNING: GPU not detected. Check CUDA installation.")

Python Version: 3.13.13 (tags/v3.13.13:01104ce, Apr  7 2026, 19:25:48) [MSC v.1944 64 bit (AMD64)]
CUDA Available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.00 GB


## Architecture

### VAE

In [41]:
from diffusers import AutoencoderKL

# 1. Clear any remaining cache
torch.cuda.empty_cache()

# 2. Load directly to GPU using device_map
# This is the most memory-efficient way to load a model
vae = AutoencoderKL.from_pretrained(
    "runwayml/stable-diffusion-v1-5", 
    subfolder="vae",
    torch_dtype=torch.float16,
    variant="fp16",
    device_map="auto" # This automatically places it on the GPU efficiently
)

print("VAE Loaded Successfully!")
print(f"VRAM used: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")

VAE Loaded Successfully!
VRAM used: 1544.00 MB


### Text Encoder

In [42]:
from transformers import CLIPTextModel, CLIPTokenizer

# Tokenizer doesn't use GPU memory, it stays on CPU
tokenizer = CLIPTokenizer.from_pretrained(
    "runwayml/stable-diffusion-v1-5", 
    subfolder="tokenizer"
)

# Text Encoder is the 'Conditioning' (y) part of the paper
text_encoder = CLIPTextModel.from_pretrained(
    "runwayml/stable-diffusion-v1-5", 
    subfolder="text_encoder",
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Text Encoder Loaded!")
print(f"Total VRAM used: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")

Loading weights: 100%|██████████| 196/196 [00:00<00:00, 700.73it/s]


Text Encoder Loaded!
Total VRAM used: 1533.59 MB


### U-Net

In [43]:
from diffusers import UNet2DConditionModel

unet = UNet2DConditionModel.from_pretrained(
    "runwayml/stable-diffusion-v1-5", 
    subfolder="unet",
    torch_dtype=torch.float16,
    variant="fp16",
    device_map="auto"
)

print("U-Net Loaded Successfully!")
print(f"Total VRAM used: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")

U-Net Loaded Successfully!
Total VRAM used: 3187.37 MB


### Scheduler

In [44]:
from diffusers import LMSDiscreteScheduler

# This matches the 'Linear Noise Schedule' mentioned in the paper
scheduler = LMSDiscreteScheduler.from_pretrained(
    "runwayml/stable-diffusion-v1-5", 
    subfolder="scheduler"
)

print("Scheduler defined.")

Scheduler defined.


## Image Generation

In [45]:
from tqdm.auto import tqdm
from PIL import Image
import numpy as np
import math
import torch

seed = 42
image_index = math.floor(10 * np.random.rand())  # Random index for saving the image

# 1. Configuration - Using CPU for the random generator to avoid CUDA unknown errors
prompt = ["A street sign that reads Ankit Mishra"]
height, width = 512, 512
steps = 50  # Number of denoising steps 
guidance_scale = 7.5  

# Create generator on CPU
generator = torch.Generator(device="cpu").manual_seed(seed) 

# 2. Encode the text prompt
text_input = tokenizer(prompt, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt")
with torch.no_grad():
    # Only move the specific tensor needed for the encoder to 'cuda'
    text_embeddings = text_encoder(text_input.input_ids.to("cuda"))[0]

# 3. Prepare Unconditional Embeddings
max_length = text_input.input_ids.shape[-1]
uncond_input = tokenizer([""], padding="max_length", max_length=max_length, return_tensors="pt")
with torch.no_grad():
    uncond_embeddings = text_encoder(uncond_input.input_ids.to("cuda"))[0]

text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

# 4. Create initial random noise in LATENT space (64x64)
# We create it on CPU first then move it to GPU
latents = torch.randn(
    (1, unet.config.in_channels, height // 8, width // 8), 
    generator=generator
).to("cuda", dtype=torch.float16)

latents = latents * scheduler.init_noise_sigma

print("Setup complete. Ready for Denoising Loop.")
print("Generating image...")
# 5. Denoising Loop
scheduler.set_timesteps(steps)

for t in tqdm(scheduler.timesteps):
    # Expand latents for classifier-free guidance
    latent_model_input = torch.cat([latents] * 2)
    latent_model_input = scheduler.scale_model_input(latent_model_input, t)

    # Predict noise residual
    with torch.no_grad():
        noise_pred = unet(latent_model_input, t, encoder_hidden_states=text_embeddings).sample

    # Perform guidance
    noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
    noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

    # Compute previous noisy sample (z_t -> z_t-1)
    latents = scheduler.step(noise_pred, t, latents).prev_sample

# 6. Decode Latent -> Image
latents = 1 / 0.18215 * latents # 0.18215 is the scaling factor from the paper
with torch.no_grad():
    image = vae.decode(latents).sample

# 7. Convert to PIL
image = (image / 2 + 0.5).clamp(0, 1)
image = image.detach().cpu().permute(0, 2, 3, 1).numpy()
image = (image * 255).astype("uint8")
final_image = Image.fromarray(image[0])
final_image.save(f"Generated_Image/result_{image_index}.png")
final_image.show()

print(f"Final VRAM used: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")

Setup complete. Ready for Denoising Loop.
Generating image...


100%|██████████| 50/50 [00:05<00:00,  8.66it/s]


Final VRAM used: 3187.53 MB


## Image Reconstruction

In [46]:
import torch
import numpy as np
#import requests         # We won't use this since we're loading from a local path
#from io import BytesIO  # We won't use this since we're loading from a local path
from skimage.metrics import peak_signal_noise_ratio as psnr_metric

random_index = math.floor(10 * np.random.rand())  # Random index for saving the image

# 1. Loading a test image
local_path = "D:/DLProject/Images&ReconstructedImages/image_2.jpg"

try:
    original_pil = Image.open(local_path).convert("RGB").resize((512, 512))
    print("Local image loaded successfully!")
except FileNotFoundError:
    print(f"Error: Could not find the image at {local_path}. Please check the folder.")

### In case we want to load from a URL instead, uncomment the following lines and provide a valid URL
# url = ""  # Replace with your local path or a valid URL
# response = requests.get(url)
# original_pil = Image.open(BytesIO(response.content)).convert("RGB").resize((512, 512))

# 2. Pre-process for VAE (Scale to -1 to 1)
img_array = np.array(original_pil).astype(np.float32) / 127.5 - 1.0
img_tensor = torch.from_numpy(img_array).permute(2, 0, 1).unsqueeze(0).to("cuda", dtype=torch.float16)

# 3. The Test: Encode -> Decode
with torch.no_grad():
    # Phase A: Spatial Compression (E)
    posterior = vae.encode(img_tensor).latent_dist
    latents = posterior.mode() # We use mode for a deterministic reconstruction
    
    # Phase B: Reconstruction (D)
    reconstruction = vae.decode(latents).sample

# 4. Post-process back to Pixels
recon_array = (reconstruction.squeeze().cpu().permute(1, 2, 0).numpy() + 1.0) * 127.5
recon_array = recon_array.clip(0, 255).astype(np.uint8)
recon_pil = Image.fromarray(recon_array)

# 5. Calculate Metrics for your "Results Section"
local_psnr = psnr_metric(np.array(original_pil), recon_array)
paper_psnr = 27.1  # Benchmark from the LDM paper for f=8
delta_percent = ((local_psnr - paper_psnr) / paper_psnr) * 100

print(f"--- RECONSTRUCTION TEST RESULTS ---")
print(f"Local PSNR: {local_psnr:.2f} dB")
print(f"Paper PSNR: {paper_psnr} dB")
print(f"Delta: {delta_percent:.2f}%")

# Save the side-by-side comparison for your Qualitative Results slide
comparison = Image.new('RGB', (1024, 512))
comparison.paste(original_pil, (0, 0))
comparison.paste(recon_pil, (512, 0))
comparison.save(f"Images&ReconstructedImages/reconstruction_comparison_{random_index}.png")
comparison.show()

Local image loaded successfully!
--- RECONSTRUCTION TEST RESULTS ---
Local PSNR: 25.28 dB
Paper PSNR: 27.1 dB
Delta: -6.72%


## Image Inpainting

In [47]:
import torch
from diffusers import StableDiffusionInpaintPipeline
from PIL import Image

random_inp_index = math.floor(10 * np.random.rand())  # Random index for saving the image

# 1. Loading the specialized Inpainting Pipeline
# We still use the same v1-5 weights, but a different pipeline class
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

# 2. Prepare images
# Make sure 'image.jpg' and 'mask.jpg' are 512x512
image_name = '6458524847_2f4c361183_k'
init_image = Image.open(f"inpainting_examples/{image_name}.png").convert("RGB").resize((512, 512))
mask_image = Image.open(f"inpainting_examples/{image_name}_mask.png").convert("RGB").resize((512, 512))

# 3. The Inpainting Prompt
prompt = "clean empty background, wall, floor"
negative_prompt="person, girl, woman, face, human, body"

# 4. Executing Inpainting
# strength tells the model how much of the original texture to ignore
output_image = pipe(
    prompt=prompt, 
    negative_prompt=negative_prompt,
    image=init_image, 
    mask_image=mask_image,
    strength=0.99, 
    guidance_scale=12
).images[0]

output_image.save(f"inpainting_results/inpainting_result_{random_inp_index}.png")
output_image.show()

100%|██████████| 49/49 [00:07<00:00,  6.15it/s]


## Super Resolution

In [53]:
import torch
from PIL import Image
import numpy as np
from diffusers import LDMSuperResolutionPipeline

# 1. Manually Load the Sub-Components
model_id = "CompVis/ldm-super-resolution-4x-openimages"
# fetch the specific U-Net and VAE trained for SR
pipe = LDMSuperResolutionPipeline.from_pretrained(
    model_id,
    use_safetensors=False, 
    low_cpu_mem_usage=False
    ).to("cuda")
unet = pipe.unet
vqvae = pipe.vqvae
scheduler = pipe.scheduler

# 2. Prepare the "Low-Res" Signal
low_res_img = Image.open("LowResImages/image_lowres.jpeg").convert("RGB").resize((128, 128))

# 3. Pre-process the image for the U-Net
# Convert to tensor, scale to [0, 1], then normalize to [-1, 1]
image_arr = np.array(low_res_img).astype(np.float32) / 255.0
image_tensor = torch.from_numpy(image_arr).permute(2, 0, 1).unsqueeze(0).to("cuda")
image_tensor = 2.0 * image_tensor - 1.0  # Crucial for LDM normalization

# 4. Initialize Latents (The Noise)
# The latents must match the target resolution (128x128 in this model)
latents = torch.randn((1, 3, 128, 128), device="cuda", dtype=torch.float32)

# 5. Denoising Loop
scheduler.set_timesteps(100)

for t in scheduler.timesteps:
    # STEP: Channel-wise Concatenation
    # Stack the noise (3ch) and the low-res guide (3ch) to make 6 channels
    model_input = torch.cat([latents, image_tensor], dim=1) 
    
    with torch.no_grad():
        # Predict the noise using the 6-channel input
        # We don't use 'conditioning' or 'class_labels' as keywords here
        noise_pred = unet(model_input, t).sample
    
    # Update latents (only the 3-channel noise part is updated)
    latents = scheduler.step(noise_pred, t, latents).prev_sample

# 6. Decode the Latents using VQ-VAE
with torch.no_grad():
    # The VQ-VAE takes the 128x128 latents and reconstructs the image
    image_high_res = vqvae.decode(latents).sample

# 7. Post-process to viewable Image
image_high_res = (image_high_res / 2 + 0.5).clamp(0, 1) # Shift back to [0, 1]
image_high_res = image_high_res.cpu().permute(0, 2, 3, 1).numpy()[0]
final_image = Image.fromarray((image_high_res * 255).astype(np.uint8))

final_image.save("LowResImages/manual_super_res.png")
final_image.show()

Loading pipeline components...: 100%|██████████| 3/3 [00:00<00:00, 32.07it/s]
